# Step 2 — Data and market regimes

**RSM434 Industry Project 3 · Option strategies through market regimes**

This notebook collects the data and classifies every month-end into a market regime, using only information available on that date. It produces the monthly table that Step 3 (metrics by regime) builds on.

1. Data sources and download
2. Data quality, bad prints, and the Cboe files vs. Bloomberg
3. Month-end levels and monthly returns
4. Checking the benchmark and credit series
5. Regime rules
6. No-look-ahead checks
7. Months per regime
8. Regime timeline
9. How persistent the regimes are
10. Save for Step 3

The rules and data handling live in `ip3.py`, so the later notebooks use exactly the same definitions.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

import ip3

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

REFRESH_DATA = False  # True re-downloads every free source into data/raw

## 1. Data sources

| Data | Source | Used for |
|---|---|---|
| BXM, BXMD, BXY, PUT, CNDR, BFLY, CLL, PPUT | Bloomberg `<symbol> Index`; the Cboe website CSVs are the fallback and a cross-check | Strategy returns |
| S&P 500 total return | Bloomberg `SPXT Index`; Yahoo `^SP500TR` is the fallback | Benchmark, beta |
| VIX | FRED `VIXCLS` | Volatility level and direction |
| 2-year and 5-year Treasury yields | FRED `DGS2`, `DGS5` | 2s5s slope signal |
| Investment-grade credit spread | Bloomberg `LUACOAS Index` (US corporate IG OAS); FRED `BAA10Y` (Moody's Baa minus 10-year Treasury) is the fallback | Credit signal |
| ICE BofA US Corporate IG OAS | FRED `BAMLC0A0CM` (last 3 years only) | Checking the credit series |
| 3-month T-bill rate | FRED `DTB3` | Risk-free rate |

**Bloomberg data.** Open `bloomberg_pull_template.xlsx` on a Bloomberg terminal and follow its README sheet. Then save the filled workbook in `data/raw/bloomberg/` (any file name) and re-run this notebook. It picks Bloomberg over the free sources wherever Bloomberg data exists and prints which source fed each series.

In [ ]:
if REFRESH_DATA or not (ip3.RAW / "manifest.json").exists():
    ip3.download_all()
manifest = json.loads((ip3.RAW / "manifest.json").read_text())

raw = ip3.load_raw()
daily, sources = ip3.build_daily(raw)
month_ends = ip3.month_end_calendar(daily)
sample = month_ends[month_ends.to_period("M") >= ip3.FIRST_READING]

print("Free sources downloaded:", manifest["downloaded_at"])
if ip3.bloomberg_workbook():
    print("Bloomberg workbook:     ", ip3.bloomberg_workbook().name)
print("Strategies:             ", sources["strategies"])
print("Benchmark:              ", sources["benchmark"])
print("Credit spread:          ", sources["credit"])
print(f"Regime readings:         {sample[0]:%d %b %Y} to {sample[-1]:%d %b %Y} ({len(sample)} month-ends)")
print(f"Measured returns:        {sample[1]:%b %Y} to {sample[-1]:%b %Y} ({len(sample) - 1} months)")

## 2. Data quality

The table covers the 12 months before the first reading through the last month-end, because the VIX median looks back 12 months. Strategy rows named by symbol are the Cboe files; `BBG_` rows are the Bloomberg feed.

- `cboe_days_missing`: Cboe trading days with no value. FRED series miss a few because the bond market closes on Columbus Day and Veterans Day; that is normal.
- `blank_values`: FRED leaves market holidays blank; also normal.
- `unchanged_days`: days with exactly no change. Normal for interest rates; for a strategy index it usually means a stale print.
- `largest_daily_%_move`: the biggest one-day move of each index. Anything well beyond the S&P 500's worst day (about −12% on 16 Mar 2020) needs a closer look.

In [ ]:
quality = ip3.data_quality(raw, sample[0] - pd.DateOffset(months=12), sample[-1])
display(quality.round(2))

### Bad prints fixed

Every fix is listed in `ip3.DATA_FIXES` with its reason, so the fixes are explicit and easy to review. With the Bloomberg feed only one is needed:

- **BFLY, 31 May 2022:** +6.8% then −5.9% the next day while the S&P 500 fell 0.6% and 0.7%. Both feeds carry it, so Bloomberg cannot confirm it either way, but a T-bill-backed iron butterfly does not jump 6.8% and fall back the next day in a quiet market. The day is dropped. It falls on a month-end, so without the fix May 2022 would be overstated and June 2022 understated by about 6.8 percentage points.

If the Bloomberg feed is missing, the notebook falls back to the Cboe files and applies the four Cboe fixes in `ip3.DATA_FIXES["cboe"]` instead.

In [ ]:
daily, fixes_log = ip3.apply_data_fixes(daily, ip3.DATA_FIXES[sources["strategy_feed"]])
display(fixes_log[["series", "date", "action", "return_before_%", "return_after_%", "reason"]].round(2))

### Cboe files vs. Bloomberg

The same eight indices come from two feeds. Comparing them day by day showed the Cboe website files have errors that Bloomberg's history does not, which is why Bloomberg is the strategy source:

- **CLL:** the Cboe file has a permanent −19.8% jump on 16 Oct 2009, when Bloomberg shows −0.7%. It also has a different history before 2010 (in Sep 2008 it loses more than the S&P 500, which a collar cannot do), stale values on many option-expiry days, and a level about 5% off from 28 Feb to 6 Mar 2020, inside the COVID case-study window.
- **BXY:** a one-day bad print on 11 Mar 2013 (−25.6% then +34.6%), when Bloomberg shows +0.2%.
- **CNDR:** the Cboe file sits about 1.3% low from 1 Nov 2016 to 12 May 2017, shifting the Nov 2016 and May 2017 returns.
- **BXM and PUT:** a few one-day timing differences that line up again the next day; on those days Bloomberg's values move with the S&P 500.
- **BXMD and PPUT:** identical apart from rounding. BFLY is too, apart from the May 2022 bad print that both feeds carry.

The table shows how far each Cboe file is from the series in use, daily and in monthly returns.

In [ ]:
if sources["strategy_feed"] == "bloomberg":
    display(ip3.compare_feeds(daily, sample).round(2))

Below are the remaining days where a strategy moved more than 4 percentage points away from what its beta to the S&P 500 implies. These were reviewed and kept. They are crisis or option-expiry days, where nonlinear option payoffs are expected to break from the market: for example, CLL barely moved in mid-March 2020 because its puts were deep in the money.

In [ ]:
display(ip3.suspicious_days(daily, sample[0], sample[-1]).round(2))

## 3. Month-end levels and monthly returns

- Each month-end is the last Cboe trading day of the month; the current, incomplete month is excluded.
- Every series takes its last value on or before that date. The tables below show how often that value is older than the month-end itself.
- Monthly return = simple return from one month-end to the next.
- Risk-free return for a month = the 3-month T-bill rate at the start of the month ÷ 12.

In [ ]:
levels, age = ip3.sample_at(daily, month_ends)
returns = ip3.monthly_returns(levels)

key_cols = [*ip3.STRATEGIES, "SPX_TR", "VIXCLS", "DGS2", "DGS5", "DTB3", "CREDIT_BP"]
sample_age = age.loc[sample, key_cols]
display(pd.DataFrame({"max days old": sample_age.max(), "month-ends not on the day": (sample_age > 0).sum()}).T)
display(sample_age[(sample_age > 0).any(axis=1)])

A value older than the month-end comes from a dropped bad print or a gap in a feed (for example BFLY at May 2022, which falls back to 27 May). The affected returns are shifted by a few days rather than lost.

## 4. Checking the benchmark and credit series

**Benchmark.** Calendar-year returns from our S&P 500 total return series, compared with the S&P 500 total returns published by S&P Dow Jones Indices.

**Credit spread.** The spread in use is compared with two others: ICE BofA's investment-grade spread (the series the brief names, which FRED only keeps for 3 years), and the Moody's Baa minus 10-year spread (the free fallback). The regime only uses the *direction* of the 3-month change, so the key numbers are how often they agree on widening vs. tightening, and in how many months switching series would change the call.

In [ ]:
published = pd.Series({
    2009: 26.46, 2010: 15.06, 2011: 2.11, 2012: 16.00, 2013: 32.39, 2014: 13.69, 2015: 1.38, 2016: 11.96,
    2017: 21.83, 2018: -4.38, 2019: 31.49, 2020: 18.40, 2021: 28.71, 2022: -18.11, 2023: 26.29, 2024: 25.02,
}, name="Published (%)")
spx = daily["SPX_TR"].dropna()
ours = (spx.groupby(spx.index.year).last().pct_change() * 100).reindex(published.index).rename("Our data (%)")
bench_check = pd.concat([ours, published], axis=1)
bench_check["Difference (pp)"] = bench_check["Our data (%)"] - bench_check["Published (%)"]
display(bench_check.round(2).T)

in_sample = returns.loc[sample[1:]]
print(f"Largest gap vs. published: {bench_check['Difference (pp)'].abs().max():.2f} pp")
print(f"Correlation of monthly returns with BXM: {in_sample['SPX_TR'].corr(in_sample['BXM']):.2f}")

In [ ]:
credit = levels.loc[sample, ["CREDIT_BP", "BAMLC0A0CM", "BAA10Y"]].copy()
credit[["BAMLC0A0CM", "BAA10Y"]] *= 100
change_3m = credit.diff(3)


def agreement(other):
    pair = change_3m[["CREDIT_BP", other]].dropna()
    same = (pair["CREDIT_BP"] > 0) == (pair[other] > 0)
    return {
        "from": f"{pair.index[0]:%b %Y}", "months": len(pair),
        "corr of 3-month changes": round(pair.corr().iloc[0, 1], 2),
        "same widening/tightening call": f"{same.mean():.0%}", "months the call differs": int((~same).sum()),
    }


print("Credit spread in use:", sources["credit"])
display(pd.DataFrame({
    "vs. ICE BofA IG OAS": agreement("BAMLC0A0CM"),
    "vs. Moody's Baa minus 10-year": agreement("BAA10Y"),
}).T)

spreads = daily.loc[sample[0]:sample[-1], ["CREDIT_BP", "BAMLC0A0CM", "BAA10Y"]].copy()
spreads[["BAMLC0A0CM", "BAA10Y"]] *= 100
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(spreads["CREDIT_BP"].dropna(), lw=1, label=f"In use: {sources['credit']}")
ax.plot(spreads["BAMLC0A0CM"].dropna(), lw=1, label="ICE BofA US Corporate IG OAS (FRED BAMLC0A0CM)")
ax.plot(spreads["BAA10Y"].dropna(), lw=1, alpha=0.7, label="Moody's Baa minus 10-year Treasury (FRED BAA10Y)")
ax.set_ylabel("bp")
ax.set_title("Investment-grade credit spread: series in use vs. alternatives")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ip3.OUTPUT / "credit_spread_check.png", dpi=150)
plt.show()

## 5. Regime rules

At each month-end *t*, using only data dated on or before *t*:

| Signal | Rule | States |
|---|---|---|
| Volatility level | VIX close vs. the median of daily VIX closes over the trailing 365 days | High / Low |
| Volatility direction | VIX close vs. the VIX at the previous month-end | Rising / Falling |
| Credit spread (main added signal) | Investment-grade OAS vs. its value 3 month-ends earlier | Widening / Tightening |
| 2s5s slope (robustness check) | (5-year − 2-year yield) vs. its value 3 month-ends earlier | Steepening / Flattening |

The first two signals give the baseline grid: **Low/Falling** (calm), **Low/Rising** (waking up), **High/Rising** (stress) and **High/Falling** (recovery). Adding the credit or slope signal splits each cell in two.

An exact tie (no change, or VIX exactly at its median) counts as Low / Falling / Tightening / Flattening; the tie counts are printed below.

The reading at month-end *t* is paired with the return over the **following** month, from *t* to *t* + 1.

In [ ]:
signals = ip3.compute_signals(daily, month_ends)
readings = signals.loc[sample]

ties = pd.Series({
    "VIX exactly at its 12-month median": (readings["vix"] == readings["vix_median_12m"]).sum(),
    "VIX unchanged over the month": (readings["vix_change_1m"] == 0).sum(),
    "Credit spread unchanged over 3 months": (readings["credit_change_3m_bp"] == 0).sum(),
    "2s5s slope unchanged over 3 months": (readings["slope_change_3m_bp"] == 0).sum(),
}, name="month-ends")
display(ties.to_frame())
missing = readings[["vol_regime", "regime_credit", "regime_slope"]].isna().any(axis=1).sum()
print("Month-ends with a missing reading:", int(missing))

latest_cols = [
    "vix", "vix_median_12m", "vix_change_1m", "credit_bp", "credit_change_3m_bp",
    "slope_2s5s_bp", "slope_change_3m_bp", "vol_regime", "credit_signal", "slope_signal",
]
display(readings[latest_cols].tail(12).round(2))

## 6. No look-ahead

1. **Truncation test.** For every month-end, rebuild the reading from daily data cut off at that date and compare it with the full-sample reading. Any difference would mean a rule used future data.
2. **Pairing test.** Every return month must be matched with the reading from the month-end immediately before it. `pair_with_next_month` raises an error if not.

In [ ]:
lookahead = ip3.lookahead_check(daily, month_ends, readings)
if lookahead.empty:
    print(f"Truncation test passed: all {len(readings)} readings are identical when rebuilt from data cut off at their date.")
else:
    print(f"Truncation test FAILED for {len(lookahead)} values:")
    display(lookahead)

paired = ip3.pair_with_next_month(returns, signals)
print(f"Pairing test passed: {len(paired)} return months, each matched with the previous month-end's reading.")
display(paired[["signal_date", "vol_regime", "credit_signal", "slope_signal", "BXM", "PUT", "SPX_TR", "RF"]].head(4))

## 7. Months per regime

These counts are return months grouped by the reading at the month-end before each one, so they are the number of observations each Step 3 result will rest on. The sample is split into two equal halves for the stability check. Watch the thinnest cells in the 8-regime tables: a result resting on a handful of months should not drive the playbook.

In [ ]:
halves = ip3.sample_halves(paired.index)
for half, months in halves.groupby(halves).groups.items():
    print(f"{half}: {months.min():%b %Y} to {months.max():%b %Y} ({len(months)} months)")

for col in ["vol_regime", "regime_credit", "regime_slope"]:
    display(ip3.regime_counts(paired, col))

## 8. Regime timeline

Shading shows the regime read at each month-end, drawn over the month that reading summarises; the return it is paired with is the following month's. The lower panels show the 3-month changes behind the two added signals.

In [ ]:
colors = {"Low/Falling": "#c7e9c0", "Low/Rising": "#fee391", "High/Rising": "#fc9272", "High/Falling": "#9ecae1"}
prev_end = month_ends[month_ends.get_indexer(sample) - 1]
vix = daily["VIXCLS"].dropna()
window = slice(prev_end[0], sample[-1])

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [2.2, 1, 1]})
ax = axes[0]
for start, end, regime in zip(prev_end, sample, readings["vol_regime"]):
    ax.axvspan(start, end, color=colors[regime], lw=0)
ax.plot(vix.loc[window], color="black", lw=0.7, label="VIX")
ax.plot(vix.rolling("365D").median().loc[window], color="purple", lw=1.3, ls="--", label="Trailing 12-month median")
ax.set_ylabel("VIX")
ax.set_title("Volatility regime read at each month-end")
handles = ax.get_legend_handles_labels()[0] + [
    Patch(color=c, label=f"{r} ({ip3.REGIME_NICKNAMES[r]})") for r, c in colors.items()
]
ax.legend(handles=handles, loc="upper right", ncol=3, fontsize=8)

for ax, col, up, down, title in [
    (axes[1], "credit_change_3m_bp", "widening", "tightening", "Credit spread, 3-month change (bp)"),
    (axes[2], "slope_change_3m_bp", "steepening", "flattening", "2s5s slope, 3-month change (bp)"),
]:
    change = readings[col]
    ax.bar(change.index, change, width=25, color=np.where(change > 0, "#d6604d", "#4393c3"))
    ax.axhline(0, color="black", lw=0.6)
    ax.set_title(f"{title}: red = {up}, blue = {down}", fontsize=10)
fig.tight_layout()
fig.savefig(ip3.OUTPUT / "regime_timeline.png", dpi=150)
plt.show()

## 9. How persistent are the regimes?

Each row shows, for a given regime at this month-end, the share of next month-ends that land in each regime (%). A strong diagonal means regimes are sticky, so a reading is still informative when you act on it a month later. If readings flip almost at random, switching strategies by regime would mostly add turnover.

In [ ]:
order = ip3.VOL_REGIMES
transitions = pd.crosstab(readings["vol_regime"], readings["vol_regime"].shift(-1), normalize="index")
transitions = transitions.reindex(index=order, columns=order)
transitions.index.name = "This month-end"
transitions.columns.name = "Next month-end"
display((transitions * 100).round(0))

## 10. Save for Step 3

`monthly_returns_with_prior_regime.csv` is the main input for Step 3. Each row is a return month, with the strategy and S&P 500 returns, the risk-free return, and the regime read at the previous month-end (`signal_date`), so metrics by regime come from a simple group-by.

In [ ]:
ip3.PROCESSED.mkdir(parents=True, exist_ok=True)
daily.to_csv(ip3.PROCESSED / "daily_clean.csv")
levels.loc[sample].to_csv(ip3.PROCESSED / "month_end_levels.csv")
readings.to_csv(ip3.PROCESSED / "month_end_regimes.csv")
paired.to_csv(ip3.PROCESSED / "monthly_returns_with_prior_regime.csv")
fixes_log.to_csv(ip3.PROCESSED / "data_fixes_applied.csv", index=False)
for path in sorted(ip3.PROCESSED.glob("*.csv")):
    print(path.relative_to(ip3.ROOT))